In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


●今回の方針

まず数字のみの特徴量を使って、予想を行う。欠損は中央値で埋め合わせする。"PassengerId"は生存したかどうかに影響するものではないので除外。

In [2]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

print(df_train.columns)
df_train.head()


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


※まずは特徴量一覧を表示。特徴量の説明

●数字の特徴量

PassengerId（乗客ID）：乗客にランダムに割り振られた通し番号
→機械の学習には必要なく、ノイズになる

Survived：予測の正解データ（ターゲット）値は0死亡、1生存

Pclass（チケットのクラス / 客室階級）：社会的・経済的ステータスを表す3段階の階級（1 ＝ 1等星・高級、2 ＝ 2等星・一般、3 ＝ 3等星・格安）。数字の 1, 2, 3 なのでそのまま使える。

Age（年齢）：

SibSp（同乗している兄弟・配偶者の数）

Parch（同乗している親・子供の数）

Fare（旅客運賃）：チケットの料金。運賃は「一部の富豪がめちゃくちゃ高い金額を払っている」ため、データが右に長く伸びた歪んだ分布（ロングテール）

●数字以外の特徴量

Name（氏名）：氏名に含まれる 「Mr.」「Miss.」「Master.」などの敬称（Title）を文字列抽出 して新しい特徴量として活用可能

Sex（性別）：数値（0 と 1）に変換する（数値エンコーディング）必要。maleかfemaleで記載

Ticket（チケット番号）：乗車券の券面番号（数字やアルファベットの混ざった文字列）。

Cabin（客室番号）：部屋番号（例：C23, E46 などの文字列）。データの7割以上が空欄（欠損値）であり、扱いが非常に難しい

Embarked（乗船した港）：どこの港から船に乗ったか（3つの港の頭文字）。C ＝ Cherbourg（シェルブール）Q ＝ Queenstown（クイーンズタウン）S ＝ Southampton（サウサンプトン）




In [3]:
from sklearn.ensemble import RandomForestClassifier

features = ["Pclass","Age","SibSp","Parch","Fare"] #数字のみの特徴量かつ目的変数に対して規則性を持ち得るもの

train_med = df_train[features].median()
df_train[features] = df_train[features].fillna(train_med)
df_test[features] = df_test[features].fillna(train_med)

X = df_train[features]
X_test = df_test[features]
y = df_train.Survived

model = RandomForestClassifier(n_estimators=100,max_depth=5,random_state=0)
model.fit(X,y)
prediction = model.predict(X_test)

output = pd.DataFrame({"PassengerId":df_test.PassengerId,"Survived":prediction})
output.to_csv("submission_csv",index=False)
